# <h1 align="center"> Anotação de variantes SNPs utilizando SnpEff

## <h2 align="justify"><span style="color: blue;">Esse script foi desenvolvido no contexto da dissertação de mestrado do aluno: Artur Gabriel Rodrigues Silva, no programa de pós graduação em Genética e Melhoramento de Plantas UFG (2025-2026), sob a orientação da professora Thannya Nascimento Soares, e coorientação de Marco Aurélio Caldas de Pinho Pessoa Filho da Embrapa Recursos Genéticos e Biotecnologia, e Rhewter Nunes da Universidade Estadual de Goiás.</span>

---

## Descrição do script

<h3 align="justify"> A anotação das variantes foi realizada utilizando o software SnpEff para anotação das variantes SNPs. Foi utilizado o arquivo FASTA contendo o genoma de referência para Dipteryx alata, o arquivo gff contendo a anotação funcional do genoma e por fim o arquivo VCF obtido a partir da pipeline de GATK+Freebayes.

O SnpEff fornece vários níveis de anotação, desde os mais simples (por exemplo, qual gene cada variante está afetando) até anotações extremamente complexas (por exemplo, se uma variante não codificante afetará a expressão de um gene). Deve-se observar que, quanto mais complexas são as anotações, mais elas dependem de predições computacionais. Essas predições podem estar incorretas, portanto os resultados do SnpEff (ou de qualquer algoritmo de predição) não devem ser aceitos cegamente; eles precisam ser analisados e validados de forma independente por experimentos laboratoriais correspondentes (wet-lab).

- `Database`
    - Para prodeceder para anotações, o SnpEff requer bancos de dados confiáveis. Atualmente o banco de dados deles possui mais de 20.000 genomas de referência. <b> Em alguns casos como o nosso precisamos criar nosso próprio banco de dados de maneira manual, que será explicado mais a frente. </b>
    
- `Documentação`
    - Consulte o seguinte link para conferir os comandos possíveis: https://pcingola.github.io/SnpEff/snpeff/commandline/ 

- `ATENÇÃO`
    - Versões mais antigas do GATK (v2.x/v3.x) possui em seu VCF o campo  INFO  pode vir com a tag EFF as versões mais atuais do GATK possuem a tag ANN. Caso seu VCF seja feito em uma versão mais antiga você pode utilizar a tag -o gatk para dizer para o SnpEff uitilizar a tag EFF. RECOMENDAÇÃO: É aconselhável gerar NOVAMENTE SEU VCF nas versões 4.x superiores para evitar possíveis conflitos com o SnpEff.
    - `O SnpEff fará anotação das variantes no campo ANN`
- `Dúvida`
    - No SnpEff, cada gene pode ter várias isoformas de transcritos, e o programa pode anotar variantes em todas elas ou apenas no chamado transcrito canônico. Usar apenas o canônico facilita a análise, porque reduz a redundância e dá consistência ao resultado, mas isso também pode ocultar efeitos importantes em isoformas alternativas. E a escolha impacta diretamente na interpretação biológica das variantes: se buscamos uma visão geral e simplificada, o canônico é útil; se queremos explorar possíveis efeitos em splicing alternativo ou isoformas específicas, precisamos considerar todos os transcritos.
- `ANN`
    - O campo ANN possui alguns subcampos muito característicos (sendo no total de 16 campos que podem ou não estarem preenchidos), confira o que diz respeito cada um em: https://pcingola.github.io/SnpEff/snpeff/inputoutput/.
- `Sequence Ontology`
    - o SnpEff usa "SO" porque está preocupado em descrever elementos da sequência (exon, intron,UTR, splice site) e tipos de variantes (missense, nonsense, synonymous, frameshift).
- `Previsão de impacto`
    - HIGH (Variante com alto impacto na proteína, trucamento, perda de função), MODERATE (Variante não disruptiva que pode alterar a eficácia das proteínas), LOW (Assume-se que são inofensivos ou pouco prováveis de mudar o comportamento proteico), MODIFIER (Variantes não codificantes ou variantes que afetam genes não codificantes, ausência de evidência de impacto).
- `Previsão de perda de função e decaimento mediado por nonsense`
    - Para inferir um LOF real é NECESSÁRIO validação laboratorial.
- `Outputs`
    - O SnpEff para além do VCF com as variantes anotadas, ele irá gerar um `snpEff_summart.html` [-stats ...], também é possível gerar algumas estatísticas a nível de gene em formato tabular `snpEff_genes.txt` (separado por tab) [-csvStats ...].


---

### 1. Preparando ambiente para trabalho

#### Criando ambiente conda com java >= 8

In [ ]:
%%bash

conda create -n snpeff_env openjdk=11
conda activate snpeff_env

#### Instalando o SnpEff

In [ ]:
%%bash

conda install -c bioconda snpeff

---

### 2. Criando seu próprio banco SnpEff

- Para construir umb anco de dados SnpEff você precisa:
    - 1. Do `genoma de referência` tipicamente um arquivo FASTA.
    - 2. Os `arquivos de anotação` contendo informação dos genes, transcritos e exons no genoma (tipicamente arquivos GTF, GeneBank, GFF, RefSeq).
    - 3. Sequências de CDS ou Proteínas: são utilizados para checar se o banco de dados é consistente e não tem erros.

#### 2.1 Configurando um novo genoma

Edite o arquivo de configuração (snpEff.config) para criar um novo genoma

In [ ]:
%%bash

vi ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/snpEff.config

Adicione as seguintes linhas dentro do snpEff.config

In [ ]:
%%bash

# Dipteryx alata genome, version 0.9
Embrapa_UFG_Dala_nuclear_0.9.genome : Dipteryx alata Vogel

#### 2.2 Construindo banco de dados a partir de arquivos GFF

O SnpEff espera que seus arquivos estejam em pastas específicas dentro da estrutura de diretórios de instalação, portanto você deve:
- `Mover seu genoma de referência e gff para:`

In [ ]:
%%bash

# Nome da pasta que o software espera estar o genoma e o arquivo GFF
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/

# Crie um diretório o gff com o mesmo nome que você colocou no snpEff.config
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/

# Movendo via soft link o arquivo gff com anotações
ln -s ~/data/projects/baru/annotation/mikado.loci.clean.step2.primaryTranscripts.gff3 ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/genes.gff

#Crie um diretório para o genoma de referência
mkdir ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/genomes/

# Movendo via soft link o genoma
ln -s ~/data/projects/baru/reference/Embrapa_UFG_Dala_nuclear_0.9.fa ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/genomes/Embrapa_UFG_Dala_nuclear_0.9.fa

- `Adicionando sequências de proteínas para validação do banco`

In [ ]:
%%bash

# Renomeie para protein.fa na hora de passar prodiretório /data/Embrapa_UFG_Dala_nuclear_0.9/
ln -s ~/data/projects/baru/annotation/mikado.loci.clean.step2.primaryTranscripts.proteins.faa ~/.conda/envs/snpeff_env/share/snpeff-5.2-3/data/Embrapa_UFG_Dala_nuclear_0.9/protein.fa

- `Criando banco de dados:`

In [ ]:
%%bash

# Se localize no seu diretório de trabalho primeiramente, e então execute:
snpeff build -gff3 -noCheckCds -v Embrapa_UFG_Dala_nuclear_0.9

Outputs: 
- sequence.bin, 
- sequence.Dal_02.bin, 
- sequence.Dal_04.bin, 
- sequence.Dal_06.bin, 
- sequence.Dal_08.bin, 
- sequence.Dal_01.bin, 
- sequence.Dal_03.bin, 
- sequence.Dal_05.bin, 
- sequence.Dal_07.bin, 
- snpEffectPredictor.bin.

---

### 4. Executando SnpEff

`Removendo loci "unplaced"`

In [ ]:
%%bash

cat allsamples.step2.tranche90.PASS.biallelic.snps.maf0.05.sitesonly.vcf | grep -v "^unplaced_" > only_chrom_snps.vcf

Descrição dos argumentos utilizados

- `ann` : formato de anotação mais moderno (Sequence Ontology)
- `-v` : Mostra progresso 
- `-stats` : HTML com gráficos e estatísticas
- `-csvStats` : CSV para análises posteriores
- `-lof` : Detecta perda de função (LOF) e NMD

In [ ]:
%%bash

snpeff ann \
  -v \
  -stats dalata_snp_prunned_stats.html \
  -csvStats dalata_snp_prunned_stats.csv \
  -lof \
  Embrapa_UFG_Dala_nuclear_0.9 \
  only_chrom_prunned_snps.vcf > snps_dalata_prunned_annotated.vcf

---

### 5. Extraindo campos para visualziação

Extraindo os campos CHROM, POS, ANN do VCF para visualização em ambiente R

In [ ]:
%%bash

bcftools query \
    -f '%CHROM\t%POS\t%INFO/ANN\n' \
    snps_dalata_annotated.vcf.gz \
    | gzip -c > ann_raw.tsv.gz

### 6. Gene Ontology das variantes

In [ ]:
%%bash

SnpSift \
    extractFields \
    -e "." snps_dalata_annotated.norepeats60.vcf \
    CHROM POS ID REF ALT \
    "ANN[0].GENE" "ANN[0].GENEID" "ANN[0].FEATUREID" \
    "ANN[0].EFFECT" "ANN[0].IMPACT" "ANN[0].HGVS_C" "ANN[0].HGVS_P" \
    > variantes_id_impact_effect.tsv

In [ ]:
%%R

---
title: "GO_analysis"
output: html_notebook
---

> Artur Silva 08/11/26

```{r}
library(tidyverse)
```

## 1. Genes com pelo menos uma variante HIGH (ANN[0])

```{r}
var <- read_tsv("variantes_id_impact_effect.tsv", skip = 1, show_col_types = FALSE,
                col_names = c("chrom","pos","id","ref","alt","gene_name","gene",
                              "transcript","effect","impact")) %>%
  filter(
    str_detect(chrom, "^Dal_0[1-8]$"),
    gene != ".",
    !str_detect(gene, "unplaced"),
    !str_detect(transcript, "unplaced")
  )
```

```{r}
genes_high <- var %>% 
  filter(impact == "HIGH") %>% 
  distinct(gene)
```

```{r}
# Resumo quantitativo dos HIGH
tibble(variantes_total = nrow(var),
       variantes_HIGH  = sum(var$impact == "HIGH"),
       genes_HIGH      = nrow(genes_high))

var %>% 
  filter(impact == "HIGH") %>% 
  count(effect, sort = TRUE)
```

## 2. Gene -> GO a partir do arquivo do eggNOG-mapper

```{r}
ann <- read_tsv("annotated_gene_ontology_terms.tsv", show_col_types = FALSE) %>%
  filter(!str_detect(query_sequence, "unplaced"))

names(ann)   # esperado: "query_sequence" "go_id" "go_term" "category"

# Quantas linhas são termos obsoletos ou termos raiz
ann %>% 
  summarise(linhas    = n(),
            obsoletos = sum(str_detect(go_term, "^obsolete")),
            raizes    = sum(go_term == category))
```

```{r}
# Transcrito -> gene, montado a partir do próprio SnpEff
tx2gene <- var %>%
  filter(transcript != ".") %>%
  distinct(query = transcript, gene)

gene2go <- ann %>%
  filter(
    !str_detect(go_term, "^obsolete"),
    !go_id %in% c("GO:0008150", "GO:0005575", "GO:0003674")
  ) %>%
  inner_join(tx2gene, by = c("query_sequence" = "query")) %>%
  distinct(gene, go_id, go_term, category)
```

#### O objeto tx2gene é montado a partir da tabela do SnpEff (var), usando as colunas transcript e gene. Depois esse objeto é unido ao arquivo de anotação
#### Ou seja, o ID do transcrito que o SnpEff reporta é comparado com o query_sequence do EnTAP. Essa é a ponte que leva do GO (anotado por transcrito) ao gene (a unidade em que você filtra HIGH).

## 3. Cobertura: quantos genes HIGH têm GO

```{r}
tibble(genes_HIGH = nrow(genes_high),
       com_GO     = sum(genes_high$gene %in% gene2go$gene))
```

## 4. Termos GO dos genes HIGH, com nome e ontologia

```{r}
go_high <- genes_high %>% 
  inner_join(gene2go, by = "gene")

tabela <- go_high %>%
  count(ONTOLOGY = category, GO = go_id, TERM = go_term, name = "n_genes") %>%
  arrange(ONTOLOGY, desc(n_genes))

write_csv(tabela, "GO_genes_HIGH.csv")
```

## 5. Gráfico: termos mais frequentes por ontologia

#### Cada barra deve ser lida individualmente, como "quantos dos meus genes HIGH têm esse termo"

```{r, fig.width=7, fig.height=11}
rotulos <- c(biological_process = "Processo biológico",
             cellular_component = "Componente celular",
             molecular_function = "Função molecular")

cores <- c(biological_process = "#0072B2",
           cellular_component = "#009E73",
           molecular_function = "#D55E00")

p <- tabela %>%
  group_by(ONTOLOGY) %>% 
  slice_max(n_genes, n = 15, with_ties = FALSE) %>% 
  ungroup() %>%
  ggplot(aes(n_genes, reorder(TERM, n_genes), fill = ONTOLOGY)) +
  geom_col(width = 0.75) +
  geom_text(aes(label = n_genes), hjust = -0.15, size = 3) +
  facet_wrap(~ONTOLOGY, scales = "free_y", ncol = 1,
             labeller = as_labeller(rotulos)) +
  scale_fill_manual(values = cores, guide = "none") +
  scale_y_discrete(labels = function(x) str_wrap(x, width = 45)) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.12))) +
  labs(x = "Nº de genes com variante HIGH", y = NULL) +
  theme_classic(base_size = 11) +
  theme(strip.background = element_blank(),
        strip.text = element_text(face = "bold", hjust = 0),
        axis.text.y = element_text(color = "black"))

p
ggsave("GO_genes_HIGH.png", p, width = 7, height = 11, dpi = 300)
ggsave("GO_genes_HIGH.pdf", p, width = 7, height = 11)
```

O arquivo do SnpEff (variantes_id_impact_effect.tsv) tem uma linha por variante, com as colunas chrom, pos, id, ref, alt, gene_name, gene, transcript, effect, impact:

| chrom  | pos  | gene   | transcript | effect             | impact   |
|---------|------|--------|------------|--------------------|----------|
| Dal_01 | 1520 | GENE_A | GENE_A.1   | stop_gained        | HIGH     |
| Dal_01 | 1788 | GENE_A | GENE_A.2   | missense_variant   | MODERATE |
| Dal_03 | 9041 | GENE_B | GENE_B.1   | frameshift_variant | HIGH     |

O arquivo do EnTAP (annotated_gene_ontology_terms.tsv) tem uma linha por par sequência-termo, com query_sequence, go_id, go_term, category, e não tem coluna de gene:

| query_sequence | go_id      | go_term                     | category            |
|----------------|------------|-----------------------------|---------------------|
| GENE_A.1       | GO:0006355 | regulation of transcription | biological_process  |
| GENE_A.1       | GO:0005634 | nucleus                     | cellular_component  |
| GENE_A.2       | GO:0005634 | nucleus                     | cellular_component  |
| GENE_B.1       | GO:0003677 | DNA binding                 | molecular_function  |

O que o script faz, passo a passo

- genes_high. Filtra impact == "HIGH" e aplica distinct(gene). Sobra uma coluna só, com GENE_A e GENE_B. A variante, a posição e o transcrito deixam de existir nessa tabela.
- tx2gene. Pega transcript e gene do SnpEff, remove duplicatas e renomeia transcript para query. O resultado é GENE_A.1 e GENE_A.2 apontando para GENE_A, e GENE_B.1 para GENE_B. O renomear serve só para o nome da coluna de junção.
- gene2go. Primeiro filtra o arquivo do EnTAP, tirando obsoletos e raízes. Depois faz o inner_join entre query_sequence e query, que compara as duas strings como estão. Cada linha do EnTAP ganha a coluna gene. 
- Por fim o distinct(gene, go_id, go_term, category) descarta a coluna query_sequence. No exemplo, “nucleus” aparece em GENE_A.1 e em GENE_A.2, mas vira uma linha só para GENE_A.
- go_high. Une genes_high e gene2go pela coluna gene.
- tabela. Conta, por ontologia e termo, quantos genes sobraram. Esse número é o n_genes das barras.

> Os genes com ao menos uma variante de alto impacto, segundo a classificação do SnpEff, foram caracterizados funcionalmente por meio de termos de Gene Ontology. Considerou-se o efeito mais severo predito para cada variante, e a análise ficou restrita aos oito cromossomos nucleares de Dipteryx alata. O SnpEff prediz o efeito das variantes sobre transcritos e associa cada transcrito ao seu gene, e o EnTAP anota os termos GO também por transcrito. Os dois conjuntos foram integrados pelo identificador do transcrito e consolidados no nível do gene, de modo que cada gene com variante HIGH recebeu os termos GO de seus transcritos, contados uma única vez por gene. Termos obsoletos e termos raiz foram descartados. Para cada categoria da ontologia, contou-se o número de genes associados a cada termo, e os 15 mais frequentes foram representados graficamente. As análises foram feitas em R.